In [2]:
# import dependencies 
import pandas as pd
import  pandas_datareader.data as web
from datetime import datetime
import yfinance as yf
import matplotlib.pyplot as plt

### Money at a fixed rate for an secured purchase: 

Datasets to obtain: 
1. **SOFR rates** from Federal Reserve Bank of New York from 2016 to 2023
2. **Interest rates** over the past 8 years


### Fetch The Data:

1. SOFT Data

In [ ]:
#import the SOFR data: 

# Define the date range
start = datetime(2016, 1, 1)
end = datetime(2023, 12, 31)

# Fetch SOFR data
SOFR_df = web.DataReader('SOFR', 'fred', start, end)
SOFR_df = SOFR_df.dropna()

# Resampling to monthly data
SOFR_df = SOFR_df.resample('M').mean().reset_index()

SOFR_df

2. Interest rate data

In [ ]:
# Fetch the The Bank Prime Lending Rate from FRED:

# Define the date range, shift dates to align with default quarters
start = datetime(2018, 4,30)
end = datetime(2023, 12, 31)

# Fetch the delinquency rate on credit cards
bank_prime_rates_df = web.DataReader('DPRIME', 'fred', start, end)

#resample the data to monthly

bank_prime_rates_df = bank_prime_rates_df.resample('M').mean()


bank_prime_rates_df.reset_index(inplace=True)

#drop the last two rows to align with the default rate data
bank_prime_rates_df = bank_prime_rates_df.drop(bank_prime_rates_df.index[-2:])
int_rates_df = bank_prime_rates_df
int_rates_df

##### Combine the dataframes

In [ ]:
#Combine the dataframes
secured_combined_df = bank_prime_rates_df.copy()
secured_combined_df.rename(columns={'DPRIME': 'Int_rates', 'DATE': 'Date'}, inplace=True)
secured_combined_df['SOFR'] = SOFR_df['SOFR']
secured_combined_df

### Analysing the data

In [ ]:
secured_combined_df.describe()

#### 1) SOFR Rates:

In [ ]:
# Visualizing the SOFR over time
plt.figure(figsize=(12, 6))
plt.plot(SOFR_df['DATE'], SOFR_df['SOFR'])
plt.xlabel('Date')
plt.ylabel('SOFR Rate')
plt.title('SOFR Rate Over Time')
plt.show()

The above graph shows the SOFR rate from 2018. We can see that the rate was relatively low during 2018 and 2019 and continued to increase to 2019

#### 2) Interest Rates:

In [ ]:
# Visualizing the Interest rates over time
plt.figure(figsize=(12, 6))
plt.plot(secured_combined_df['Date'], secured_combined_df['Int_rates'])
plt.xlabel('Date')
plt.ylabel('SOFR Rate')
plt.title('SOFR Rate Over Time')
plt.show()

#### 3. Correlation Analysis

In [ ]:
# Correlation analysis
import seaborn as sns

# Calculating corr matrix
corr_matrix = secured_combined_df[['Int_rates', 'SOFR']].corr()

# Set the size of the plot
plt.figure(figsize=(8, 6))

# Create a heatmap
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", square=True)

# Title and show the plot
plt.title('Correlation Heatmap')
plt.show()

print(secured_combined_df.corr())

### Publically Traded Equity:

#### Datasets to obtain:
1. Historical performance of **Netflix** stock over the past 8 years, from 2016 to 2023.
2. Historical performance of the **S&P 500** over the past 8 years, from 2016 to 2023.


### Fetching the data 

1. Netflix price data

In [ ]:
#Get Netflix price data from the yfinance

# Download historical data of Netflix
netflix_df = pd.DataFrame(yf.download("NFLX", start='2016-01-01', end='2023-12-31')).reset_index()
netflix_df = netflix_df[["Date", "Close"]]
netflix_df = netflix_df.groupby(pd.Grouper(key='Date', freq='M'))['Close'].mean()
netflix_df.reset_index()

# Resample to Monthly and calculate the stock prices
netflix_df = netflix_df.resample('M').mean().reset_index()
netflix_df


2. S&P 500 price data

In [ ]:
#Get S&P500 data from the yfinance

# Download historical data of Netflix
SP_df = pd.DataFrame(yf.download("^GSPC", start='2016-01-01', end='2023-12-31')).reset_index()
SP_df = SP_df[["Date", "Close"]]
SP_df = SP_df.groupby(pd.Grouper(key='Date', freq='M'))['Close'].mean()
SP_df.reset_index()

# Resample to Monthly and calculate the prices
SP_df = SP_df.resample('M').mean().reset_index()
SP_df

In [ ]:
#combine the three dataframes into one dataframe for analysis

Equity_combined_df = SP_df.copy()
Equity_combined_df.rename(columns={'Close': 'S&P_500'}, inplace=True)
Equity_combined_df['Netflix'] = netflix_df['Close'] 
Equity_combined_df

#### Analyze the Data:


In [ ]:
Equity_combined_df[['Netflix', 'S&P_500']].describe()

#### Netflix Stock:

In [ ]:
# Visualizing the price of netflix over time
plt.figure(figsize=(12, 6))
plt.plot(Equity_combined_df['Date'], Equity_combined_df['Netflix'])
plt.xlabel('Date')
plt.ylabel('Netflix price')
plt.title('Netflix stock price change over time')
plt.show()

#### S&P 500:

In [ ]:
# Visualizing the S&P500 over time
plt.figure(figsize=(12, 6))
plt.plot(Equity_combined_df['Date'], Equity_combined_df['S&P_500'])
plt.xlabel('Date')
plt.ylabel('S&P 500')
plt.title('S&P500 over time')
plt.show()

#### Correlation Analysis on Netflix & S&P500:

In [ ]:
# Calculating corr matrix
corr_matrix2 = Equity_combined_df[['S&P_500', 'Netflix']].corr()

# Set the size of the plot
plt.figure(figsize=(8, 6))

# Create a heatmap
sns.heatmap(corr_matrix2, annot=True, cmap='coolwarm', fmt=".2f", square=True)

# Title and show the plot
plt.title('Correlation Heatmap')
plt.show()

print(Equity_combined_df.corr())